# 05 — Centroidal力学

## 目的
NMPCが予測する「胴体と関節の縮約力学」を、接触力の合力・合momentから理解する。
本実装の既定は単一剛体だけではなく `FullCentroidalDynamics` であり、関節運動の影響を残す。

実装対応:
- `legged_interface/src/LeggedInterface.cpp`
- OCS2 `LeggedRobotDynamicsAD`（外部。完全な成分ODEはこのworkspaceでは未照合）


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        ROOT = candidate
        break

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


接触力 \(f_i\) とCoMから足へのlever arm \(r_i\) に対して
\[
m\dot v=\sum_i f_i+m g,\qquad
\dot L=\sum_i r_i\times f_i.
\]
静止なら \(\sum f_{i,z}=mg\)。4脚等分は初期guessにはなるが、加速・姿勢moment・接触数が
変われば一般には等分でない。


In [2]:
mass = 12.5
g = np.array([0.0, 0.0, -9.81])
feet = np.array([
    [ 0.25,  0.15, -0.30],  # LF
    [ 0.25, -0.15, -0.30],  # RF
    [-0.25,  0.15, -0.30],  # LH
    [-0.25, -0.15, -0.30],  # RH
])
forces = np.tile(np.array([0, 0, mass*9.81/4]), (4, 1))

net_force = forces.sum(axis=0) + mass*g
net_moment = np.cross(feet, forces).sum(axis=0)
print("net force incl. gravity [N]:", net_force)
print("net moment about CoM [N m]:", net_moment)
assert np.allclose(net_force, 0)
assert np.allclose(net_moment, 0)


net force incl. gravity [N]: [0. 0. 0.]
net moment about CoM [N m]: [0. 0. 0.]


In [3]:
# 対角2脚支持で前進加速度0.5 m/s^2を作る例。
contact = np.array([1, 0, 0, 1], dtype=bool)
forces_trot = np.zeros((4, 3))
forces_trot[contact, 0] = mass * 0.5 / contact.sum()
forces_trot[contact, 2] = mass * 9.81 / contact.sum()
a_com = forces_trot.sum(axis=0) / mass + g
moment = np.cross(feet, forces_trot).sum(axis=0)
print("forces [N]:\n", forces_trot)
print("CoM acceleration [m/s^2]:", a_com)
print("moment [N m]:", moment)


forces [N]:
 [[ 3.125   0.     61.3125]
 [ 0.      0.      0.    ]
 [ 0.      0.      0.    ]
 [ 3.125   0.     61.3125]]
CoM acceleration [m/s^2]: [0.5 0.  0. ]
moment [N m]: [ 0.    -1.875  0.   ]


## Full centroidal と SRBD
`centroidalModelType=0` はfull centroidal。状態は正規化centroidal momentumと
base pose、joint anglesで、入力後半のjoint velocityを通して形状変化も予測へ入る。
SRBDへ切替える設定値はあるが、A1既定ではない。

### 数式変更前の検査
1. 静止4脚で重力が相殺される。
2. 左右対称力でroll momentが0。
3. 接触していない脚の力が0。
4. 単位をN、N m、kg、m、sへ統一。


## 章固有の背景
                NMPCは18自由度の全運動をそのまま積分せず、全身の運動量と形状へ縮約して未来を予測する。

                ## 章固有の目的
                接触力からlinear/angular momentum rateが生じる式と、A1既定full-centroidal選択を確認する。

                ## この章のASCIIデータフロー
                ```text
                x=[h/m,base pose,q], u=[four forces,dq]
        -> LeggedRobotDynamicsAD / centroidal model
        -> xdot=[sum(F)/m+g, sum(r x F)/m, pose rate, dq]
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_interface/src/LeggedInterface.cpp
centroidalModelInfo = createCentroidalModelInfo(...); // task.info modelType=0
dynamicsPtr.reset(new LeggedRobotDynamicsAD(...));     // xdot=f(x,u)
// faithful equation map (OCS2内部成分実装はこのworkspaceで未照合)
hDot_linear = sum_i(f_i) + m*g;       // m*vdot = Σf + mg
hDot_angular = sum_i(r_i.cross(f_i)); // Ldot = Σ(r_i×f_i)
qDot_joint = u.tail(12);              // 入力後半は関節速度
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                合力が並進、CoMまわりの合momentが角運動量を変える。A1既定はSRBDではなくfull centroidalである。
